# 背景介绍与卷积数学原理

卷积运算（Convolution）是深度学习尤其是计算机视觉领域中最为核心的基础运算。自1998年LeNet-5问世以来，卷积神经网络（CNN）已发展出AlexNet、VGG、ResNet、EfficientNet等一系列里程碑式架构，在图像分类、目标检测、语义分割等任务中展现出卓越性能。昇腾AI处理器针对卷积运算进行了深度优化，Ascend C框架提供了高效的卷积计算接口，使开发者能够充分发挥硬件计算潜能。

本节学习大纲如下：
- 1.1 应用场景
- 1.2 相关网络
- 1.3 梯度
- 1.4 链式法则
- 1.5 img2col
- 1.6 正向卷积
- 1.7 反向卷积
- 1.8 扩展
- 1.9 章节测验

---

## 1.1 应用场景

### 1.1.1 卷积运算的数学基础

卷积概念源于信号处理领域，是一种将两个函数通过特定方式组合产生新函数的数学运算。在连续信号处理中，卷积的数学表达式为：

$$(f * g)(t) = \int_{-\infty}^{\infty} f(\tau)g(t-\tau)d\tau$$

在离散信号处理中，卷积定义为：

$$(f * g)[n] = \sum_{m=-\infty}^{\infty} f[m]g[n-m]$$

深度学习中的卷积运算与信号处理中的卷积存在本质区别：信号处理中的卷积需要对滤波器进行翻转操作，而深度学习中的卷积实际上是**互相关运算（Cross-correlation）**，即不进行翻转，直接在输入数据上滑动滤波器并计算局部区域的加权求和。

<img src="./images/convolution.webp" alt="卷积运算示意" width="700px">

**图示说明**：上图展示了卷积运算的基本原理。滤波器（卷积核）在输入数据上滑动，在每个位置计算局部区域与卷积核的加权求和。这种操作能够有效提取数据的局部特征，是图像处理和特征提取的核心机制。

### 1.1.2 核心应用领域

凭借其高效且自动化的特征提取能力，卷积运算在深度学习领域获得了广泛应用。以下从图像分类、目标检测、自然语言处理和视频处理四个维度展开阐述。

#### 图像分类

图像分类是卷积神经网络最经典的应用场景。卷积神经网络通过层级化的卷积操作，实现从低级特征（边缘、纹理）到高级特征（轮廓、形状、语义概念）的逐层抽象，最终经由全连接层完成分类决策。

在经典的图像分类模型如 AlexNet、VGGNet 和 ResNet 中，卷积操作都是核心模块，能够有效地提取输入图像的特征，显著提高分类准确率。以AlexNet为例，该网络是首个在大规模图像分类任务中取得突破性成果的深度卷积网络。其网络架构如下图所示，包含5个卷积层（其中第1、2、5层后接最大池化层）和3个全连接层。每个卷积层集成卷积核、偏置项、ReLU激活函数和局部响应归一化（LRN）模块，通过多层卷积逐渐提取输入图像中的不同层次特征，从低级的边缘和纹理到高级的形状和轮廓。在每一层卷积操作之后，通常会加入非线性激活函数（例如 ReLU）来增强模型的表达能力。此外，使用池化（Pooling）操作进一步减少特征图的尺寸，从而降低计算成本。其输入为$224 \times 224 \times 3$的RGB图像（部分实现填充至$227 \times 227 \times 3$），最终通过Softmax层输出分类概率。

<img src="./images/alexnet.png" alt="AlexNet网络架构" width="700px">

#### 目标检测

目标检测任务要求模型不仅能识别图像中物体的类别，还需精确定位物体的边界框位置。YOLO系列模型是该领域的代表架构，其核心思想是将目标检测转化为回归问题，通过单次前向传播同时完成分类和定位。如下图所示为YOLO系列网络结构，利用卷积操作提取图像的空间特征，通过特征金字塔结构实现多尺度目标检测，在保证实时性的同时达到较高的检测精度。该架构广泛应用于自动驾驶、智能安防、工业质检等实际场景。

<img src="./images/yoyox.png" alt="YOLOX网络架构" width="700px">

#### 自然语言处理

尽管卷积操作最初应用于图像处理领域，但其在自然语言处理中同样展现出强大的建模能力。文本数据可表示为词向量矩阵，卷积核通过滑动窗口提取局部语义特征，捕获n-gram级别的语言模式。

Text-CNN是文本分类领域的经典模型，其网络架构如下所示，通过并行设置多种尺寸（如 2、3、4）的卷积核，分别捕获不同粒度的语义单元：小尺寸卷积核聚焦于词对或三词短语的局部关系，大尺寸卷积核则倾向于提取更长的短语或从句结构。多个卷积核的输出经全局最大池化后拼接，形成固定维度的句子表示，最终通过全连接层完成分类。在情感分析、新闻分类、意图识别等任务中取得优异表现。相较于循环神经网络（RNN），卷积网络具有天然的并行计算优势，更适合处理大规模文本数据。Text-CNN的简洁设计和高效计算使其成为文本分类任务的基准模型。

<img src="./images/Text-CNN.png" alt="Text-CNN网络架构" width="700px">

#### 视频处理

视频数据相比静态图像增加了时间维度，传统2D卷积难以有效建模时序信息。3D卷积（3D Convolution）通过在空间维度（高度、宽度）和时间维度上同时滑动卷积核，实现时空特征的联合提取，适用于动作识别、视频分割、视频生成等任务。

**2D卷积与3D卷积对比**：

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <td align="left"><strong>对比项</strong></td>
  <td align="left"><strong>2D卷积</strong></td>
  <td align="left"><strong>3D卷积</strong></td>
</tr>
<tr>
  <td align="left">滑动维度</td>
  <td align="left">仅空间维度(H, W)</td>
  <td align="left">空间+时间维度(H, W, D)</td>
</tr>
<tr>
  <td align="left">输入数据</td>
  <td align="left">单帧图像</td>
  <td align="left">连续帧序列</td>
</tr>
<tr>
  <td align="left">特征提取</td>
  <td align="left">空间特征</td>
  <td align="left">时空联合特征</td>
</tr>
<tr>
  <td align="left">适用任务</td>
  <td align="left">图像处理</td>
  <td align="left">视频动作识别</td>
</tr>
</table>

2D卷积仅在空间维度滑动，处理单帧图像，无法捕获动作的时间演变信息；3D卷积则在空间和时间维度同时滑动，能够有效提取动作的时空特征。C3D、I3D等架构利用3D卷积在视频动作识别领域取得显著成果。

---

## 1.2 相关网络

卷积神经网络的发展历程可概括为三个阶段：**基础架构奠基期**（1998-2012年）、**深度突破期**（2012-2015年）和**效率优化期**（2015年至今）。每个阶段的代表性网络架构解决了当时面临的核心挑战，推动了深度学习技术的持续演进。

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <td align="left"><strong>网络名称</strong></td>
  <td align="left"><strong>年份</strong></td>
  <td align="left"><strong>核心贡献</strong></td>
</tr>
<tr>
  <td align="left">LeNet-5</td>
  <td align="left">1998</td>
  <td align="left">奠定CNN基础架构，首次成功应用于手写数字识别</td>
</tr>
<tr>
  <td align="left">AlexNet</td>
  <td align="left">2012</td>
  <td align="left">引入ReLU激活函数和Dropout，赢得ImageNet竞赛，开启深度学习热潮</td>
</tr>
<tr>
  <td align="left">VGGNet</td>
  <td align="left">2014</td>
  <td align="left">验证小卷积核(3×3)堆叠策略，网络深度达16-19层</td>
</tr>
<tr>
  <td align="left">GoogLeNet</td>
  <td align="left">2014</td>
  <td align="left">提出Inception模块，实现多尺度特征并行提取</td>
</tr>
<tr>
  <td align="left">ResNet</td>
  <td align="left">2015</td>
  <td align="left">引入残差连接，突破深层网络训练瓶颈，深度达100+层</td>
</tr>
<tr>
  <td align="left">MobileNet</td>
  <td align="left">2017</td>
  <td align="left">提出深度可分离卷积，大幅降低计算量，适配移动端部署</td>
</tr>
<tr>
  <td align="left">EfficientNet</td>
  <td align="left">2019</td>
  <td align="left">提出复合缩放策略，协同优化网络深度、宽度、分辨率</td>
</tr>
</table>

从LeNet-5奠定"卷积-池化-全连接"的基础范式，到AlexNet证明深层网络在大规模视觉任务中的有效性，再到VGGNet和GoogLeNet探索深度与宽度的平衡，ResNet通过残差连接突破深度限制，最终MobileNet和EfficientNet实现效率与性能的协同优化。整个演进路径体现了从"追求深度"到"追求效率"再到"追求均衡"的技术发展脉络。理解这些架构演进脉络，有助于开发者在昇腾平台上选择合适的网络结构并进行针对性优化。

---

## 1.3 梯度

在学校里，我们都学习过“控制变量法”:当一个结果由多个因素决定时，为了研究其中一个因素的影响，我们需要把其他因素固定不变。偏导数（Partial Derivative），就是控制变量法在微积分中的体现。

假设一个长方形的面积 $Area = Length \times Width$。面积由长和宽共同决定。
- 如果我们想知道“在宽度不变的情况下，长度变化对面积有多大影响”，这就是求面积对长度的偏导数，记作 $\frac{\partial Area}{\partial Length}$。此时，宽度被视为一个常数。
- 同理，如果我们想知道“在长度不变的情况下，宽度变化对面积的影响”，这就是求面积对宽度的偏导数 $\frac{\partial Area}{\partial Width}$。

在深度学习中，假设输出 $Y$ 是由输入 $X$ 和权重 $W$ 共同计算得出的， 比如 $Y = X * W$。当我们进行反向传播时，就会使用偏导数的思想：
- 计算 $W$ 的偏导时，我们把输入 $X$ 视作固定常数（求 $\frac{\partial Y}{\partial W}$），此时 $W$ 的偏导就是 $X$。
- 计算 $X$ 的偏导时，我们把权重 $W$ 视作固定常数（求 $\frac{\partial Y}{\partial X}$），此时 $X$ 的偏导就是 $W$。

理解了偏导数，我们在来看梯度（Gradient）的概念。
在神经网络中，偏导数描述了误差函数在单个参数上的变化率。而梯度，则是包含所有偏导数信息的集合。

我们来看一个包含多个变量的线性函数例子：$$y = ax + bw $$在这个场景中，假设 $a$ 和 $b$ 是已知的常数系数，而 $x, w$ 是我们可以调整的独立变量（在神经网络中，它们可能是输入特征或模型权重）。
- 第一步：分别计算偏导数，当我们研究 $y$ 对 $x$ 的变化率时，把 $w$ 视为常数：$\frac{\partial y}{\partial x} = a$当我们研究 $y$ 对 $w$ 的变化率时，把 $x$ 视为常数：$\frac{\partial y}{\partial w} = b$
- 第二步：组合成梯度，此时我们把求得的所有偏导数打包在一起，就得到了函数 $y$ 关于变量 $x, w$ 的梯度（数学上通常记作 $\nabla y$）：$$\nabla y = \left[ \frac{\partial y}{\partial x}, \frac{\partial y}{\partial w} \right] = [a, b]$$

这个梯度$[a, b]$指向函数值 $y$ 增加最快的方向。深度学习中，优化器会引导着网络参数逆着梯度的方向前进，从而使得误差降低，这一过程即为梯度下降。


---

## 1.4 链式法则

要理解反向传播，除了偏导数，还需要引入链式法则（Chain Rule）的概念。在微积分中，如果变量 $Y$ 依赖于 $X$，而误差函数 $L$ 依赖于 $Y$，那么 $L$ 对 $X$ 的偏导数可以通过 $Y$ 作为桥梁来计算：$$\frac{\partial L}{\partial X} = \frac{\partial L}{\partial Y} \cdot \frac{\partial Y}{\partial X}$$
想象有三个咬合在一起的齿轮：X、Y、Z。
- 当把齿轮 X 转动 1 圈时，齿轮 Y 会转动 3 圈（放大 3 倍）。
- 当齿轮 Y 转动 1 圈时，齿轮 Z 会转动 2 圈（放大 2 倍）。 

那么把齿轮 X 转动 1 圈时，齿轮 Z 转了多少圈？非常直观，答案是 $3 \times 2 = 6$ 圈。这就是链式法则的核心：嵌套函数的变化率，等于各个中间环节变化率的连乘积。

$$\frac{dz}{dx} = \frac{dz}{dy} \cdot \frac{dy}{dx} = 2 \cdot 3 = 6$$

在深度学习中，神经网络本质上也就是成千上万个这样的“齿轮”。偏导数提供了单个计算节点内梯度的处理方式，而链式法则用于解决整个网络里梯度跨层传递的问题。在训练过程中，数据正向流动得出预测结果，随后计算出与真实标签的误差（Loss，记为 $L$）。反向传播的过程，就是利用链式法则，把这个误差一层一层地往回传递，算出每个参数对最终误差该负多大责任（即计算梯度）。

对于一个标准的卷积层而言，当误差 $L$ 从网络的后端传导至当前层时，当前层会接收到来自后一层的输出梯度（通常记为 $\frac{\partial L}{\partial Y}$ 或 dY）。此时，当前卷积层必须完成两个独立的反向计算任务：
1. 计算权重的梯度（DW，Filter Gradient）： 评估卷积核参数 $W$ 对误差的责任（$\frac{\partial L}{\partial W}$）。这是反向传播的核心目的之一，算出的 DW 将被优化器用来更新网络参数，减少误差。
2. 计算输入的梯度（DX，Data Gradient）： 评估输入特征图 $X$ 对误差的责任（$\frac{\partial L}{\partial X}$）。当前层不能把误差吞掉，它必须把这部分责任继续向网络的前端传递，让前面的层也能进行更新。

### 1.4.1 梯度计算实践

我们用一个稍复杂一些的例子来加深下梯度和链式法则的理解

假设我们有一个包含两个计算节点的网络，采用基础的线性函数：
- 节点 1： $H = W_1 \cdot X + B_1$
- 节点 2： $Y = W_2 \cdot H + B_2$

为了方便处理，我们可以将其中的每个计算操作拆分出来，形成一张计算图，然后使用链式法则从后往前逐个计算参数的偏导得到梯度
<img src="./images/computational-graph.svg" >

1. 网络最终的输出为Y，我们令这一输出最后产生的初始梯度为$dY$
2. 节点 2 接收到回传的梯度 $dY$，根据计算图，使用链式法则，我们从最后一个操作开始逐步计算梯度
    - 网络最后一个操作$Y=W_2 \cdot H +B_w$是加法操作，加法产生的导数为1，所以其左右项$B_2$和$W_2 \cdot H$的偏导维持都是1，结合链式法则：其对$Y$的偏导都为$1*dY=dY$
    $$d(W_2\cdot H)=dY \cdot 1 $$ 
    $$dB_2=dY \cdot 1$$
    - 接下来的一个操作是乘法$W_2 \cdot H$，对 $W_2$ 来说其偏导是$H$ ，对 $H$ 来说其偏导是$W_2$，结合链式法则：
    $$dW_2=d(W_2\cdot H) \cdot H =dY \cdot H $$ 
    $$dH=d(W_2\cdot H)\cdot W_2 =dY \cdot W_2$$

    - 接下来的操作又是一个加法操作$H=W_1 \cdot X + B_2$，加法产生的导数为1，结合链式法则：
    $$d(W_1\cdot X)=dH \cdot 1 =dY \cdot W_2$$ 
    $$dB_1=dH \cdot 1=dY \cdot W_2$$

    - 然后是最后一个乘法$W_1 \cdot X$， 依照之前的方式可得：
    $$dW_1=d(W_1\cdot X) \cdot X =dY \cdot W_2 \cdot X $$ 
    $$dX=d(W_1\cdot X)\cdot W_1 =dY \cdot W_2 \cdot W_1$$

将整个计算逻辑也画成一张图，可以得到于正向流程相似的反向计算图
<img src="./images/computational-graph-backprop.svg" >

---

## 1.5 img2col

在理解了梯度的逐级传递后，我们面临着一个严峻的工程问题：如何在 AI Core 上高效地计算这些梯度？昇腾 AI Core 中的 Cube 单元是专为矩阵乘法设计的。传统的卷积操作采用的是“滑动窗口”模式，这种多层嵌套循环的内存访问模式极度不连续，无法发挥 Cube 单元的矩阵运算优势。

因此，在底层算子实现中，无论是正向卷积还是反向梯度计算，我们都会采用一种名为 Im2Col (Image to Column) 的经典空间转换算法，将卷积运算展开成标准的矩阵乘法。

2.1 为什么需要 Im2Col？

假设我们有一个单通道的输入特征图 $X$ 和一个卷积核 $W$。
- 滑动窗口的计算模式是：取 $X$ 的一个局部小块 $\rightarrow$ 与 $W$ 逐元素相乘并求和 $\rightarrow$ 滑动到下一个位置。这种计算并不适配cube上的矩阵运算，会导致硬件算力利用不充分
- 为了激活cube单元的算力，我们必须把卷积包装成矩阵乘法，Im2Col 的核心思想是“以空间换算力”。它将卷积核在输入图像上每次滑动覆盖的局部数据块，直接提取、展平并拼接成一个巨大的二维矩阵。

通过 Im2Col 操作，原本复杂的、带有空间位置依赖的滑动点积，瞬间被等效转换成了标准的、大规模的二维矩阵乘法。这使得卷积运算完美契合了 Cube 单元的，让硬件得以全速运转。

2.2 Img2Col工作原理
首先我们先回顾常规卷积的执行方式：

<img src="./images/conv_2.svg" >

我们有一个通道为2，3x3的fmap，以及2x2的kernel。
以结果第一个点56为例，我们的计算过程为$0*0+1*1+3*2+4*3+1*1+2*2+3*4+4*5=56$
那么为了得到整个结果，我们可以将计算过程转变成一个矩阵乘，将每个滑动窗口合并成矩阵中的一行，将每个通道kernel合并成一个向量，我们就得到了如下的矩阵过程。

<img src="./images/conv_mmad_2.svg" >

将原始数据按照滑窗展开成矩阵的过程即是img2col，通过img2col，卷积的滑窗计算变换为了矩阵乘法，从而能在cube上高效计算，我们可以使用torch的unfold接口执行img2col


In [ ]:
import torch
unfold = torch.nn.Unfold(kernel_size=(2,2))
fmap = torch.tensor(
    [1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0, 
    0.0,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0])
#转变成N,C,H,W格式
fmap=fmap.reshape(1,2,3,3)
unfold(fmap)
# 输出
# tensor([[[1., 2., 4., 5.],
#          [2., 3., 5., 6.],
#          [4., 5., 7., 8.],
#          [5., 6., 8., 9.],
#          [0., 1., 3., 4.],
#          [1., 2., 4., 5.],
#          [3., 4., 6., 7.],
#          [4., 5., 7., 8.]]])

2.3 AscendC中的Img2Col
AscendC中Img2Col时通过Load3D接口实现的，能够将L1上的张量展开到L0中，利用硬件电路完成随路重排，为了理解Load3D的行为，我们结合AscendC中文档中的两张图来解析
<img src="./images/ascendc_load3d0.png">

为了逼近总线理论带宽并最大化向量化/矩阵化计算指令的执行效率，昇腾芯片在内存访问上规定了 32 Bytes（256-bit） 的最小存取粒度。这一物理约束在昇腾架构中被抽象为不可分割的 C0 维度。
- 对于 fp16/bf16（单精度 2 Bytes），单次最小访存可吞吐 16 个标量，故 $C0 = 16$。
- 对于 fp32（单精度 4 Bytes），则 $C0 = 8$。
因此，Load3D 指令要求fmap在 L1 Cache 上的数据排布形式必须预先转换为 NC1HWC0 格式。这样能确保在提取数据时，单次读取能满足32Bytes的约束。

因此，Load3D 指令要求输入特征图在 L1 Cache 上的数据排布形式必须预先转换为 NC1HWC0 格式。这样能确保在深度方向上提取数据时，单次读取完美命中 32 Bytes 的缓存行（Cache Line）。
结合上图：
假设卷积核尺寸 $Hk=3, Wk=3$，输入通道数 $C=4$，为便于图示假设物理粒度 $C0=4$。此时 $C1 = C/C0 = 1$。
按此物理约束，展开后单行向量的真实排布为 $(1, 3, 3, 4)$，共计 36 个标量元素。观察图右侧的展开矩阵，每 4 个连续且同色的方块（如连续 4 个红色的 0），即代表一次满足 32 Bytes 硬件对齐的最小访存单元。

<img src="./images/ascendc_load3d1.png">

在大规模特征图处理场景下，完整的 Img2Col 展开会导致数据体积呈倍数级爆炸。而 AI Core 内部直连 Cube 单元的 L0 Buffer 作为极高速片上 SRAM，容量极为受限（通常为数十 KB 级别），根本无法驻留完整的展开矩阵。

上图可以看到，Load3D能够指定搬运的区间，第一次展开时我们只展开前4个滑窗的头4个元素，也就是完整img2col矩阵的左上角4*4区间，第二次则是在这一基础上往右继续展开4个元素
在 Ascend C 的 Load3D 接口层，这种切片通过下列参数控制
- mStartPt / mExtension：定义输出矩阵在 $M$ 维度（行方向） 的基地址偏移与提取长度。，即控制着当前运算覆盖 $Ho \times Wo$ 输出平面的哪一个特定区域。
- kStartPt / kExtension：定义输出矩阵在 $K$ 维度（列方向） 的基地址偏移与提取长度。即控制着 $C1 \times Hk \times Wk \times C0$ 这个深度维度的展开范围。


---

## 1.6 正向卷积

### 1.6.1 数学推导

#### 卷积的核心参数

卷积运算涉及以下核心参数，这些参数决定了卷积操作的具体计算方式：

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <td align="left" style="width: 150px"><strong>参数名称</strong></td>
  <td align="left" style="width: 220px"><strong>符号</strong></td>
  <td align="left"><strong>说明</strong></td>
</tr>
<tr>
  <td align="left">步长（stride）</td>
  <td align="left">$s_H, s_W$</td>
  <td align="left">卷积核在输入上滑动的步长</td>
</tr>
<tr>
  <td align="left">填充（padding）</td>
  <td align="left">$P_{top}, P_{bottom}, P_{left}, P_{right}$</td>
  <td align="left">在输入特征图边缘填充的像素数</td>
</tr>
<tr>
  <td align="left">空洞率（dilation）</td>
  <td align="left">$d_H, d_W$</td>
  <td align="left">卷积核元素之间的间隔，用于扩大感受野</td>
</tr>
<tr>
  <td align="left">分组数</td>
  <td align="left">$groups$</td>
  <td align="left">将输入通道和输出通道分成若干组进行独立卷积</td>
</tr>
<tr>
  <td align="left">偏置（bias）</td>
  <td align="left">$b$</td>
  <td align="left">偏置参数</td>
</tr>
</table>

#### 内存布局

NHWC 和 NCHW 是卷积神经网络（CNN）中广泛使用的数据格式。它们决定了多维数据（如图像、点云或特征图）如何存储在内存中。

**NHWC格式**（样本数、高度、宽度、通道）：
- 数据维度顺序为 $(N, H, W, C)$
- 通道维度在最后
- TensorFlow 框架及大部分推理引擎的默认格式
- 特点：同一像素点的所有通道数据连续存储，适合访存局部性优化

**NCHW格式**（样本数、通道、高度、宽度）：
- 数据维度顺序为 $(N, C, H, W)$
- 通道维度在最前
- PyTorch 框架、ONNX、Caffe 等的默认格式
- 特点：同一通道的所有数据连续存储，适合批量处理和SIMD优化

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <td align="left"><strong>对比项</strong></td>
  <td align="left"><strong>NHWC</strong></td>
  <td align="left"><strong>NCHW</strong></td>
</tr>
<tr>
  <td align="left">维度顺序</td>
  <td align="left">$(N, H, W, C)$</td>
  <td align="left">$(N, C, H, W)$</td>
</tr>
<tr>
  <td align="left">默认框架</td>
  <td align="left">TensorFlow、推理引擎</td>
  <td align="left">PyTorch、ONNX、Caffe</td>
</tr>
<tr>
  <td align="left">内存布局</td>
  <td align="left">同一像素的各通道数据连续</td>
  <td align="left">同一通道的各像素数据连续</td>
</tr>
<tr>
  <td align="left">访存特点</td>
  <td align="left">适合访问单个像素所有通道</td>
  <td align="left">适合访问整个通道平面</td>
</tr>
<tr>
  <td align="left">典型应用</td>
  <td align="left">GPU推理、图像处理</td>
  <td align="left">CPU训练、NPU计算</td>
</tr>
</table>

**示例说明**：
假设输入数据为 $2 \times 3 \times 4 \times 4$（2个样本，3通道，高4，宽4）：
- **NCHW**：内存中先存储样本0的通道0所有像素，然后是通道1所有像素，依次类推
- **NHWC**：内存中先存储样本0的像素(0,0)的3个通道值，然后是像素(0,1)的3个通道值，依次类推

#### Shape计算

**输入特征图**：$X \in \mathbb{R}^{N \times C_{in} \times H \times W}$

**卷积核**：$K \in \mathbb{R}^{C_{out} \times C_{in} \times k_H \times k_W}$

**输出特征图**：$Y \in \mathbb{R}^{N \times C_{out} \times H_{out} \times W_{out}}$

其中：
- $N$：batch size（批次大小）
- $C_{\text{in}}$：输入通道数
- $C_{\text{out}}$：输出通道数
- $H, W$：输入特征图的高度和宽度
- $k_H, k_W$：卷积核的高度和宽度
- $H_{\text{out}}, W_{\text{out}}$：输出特征图的高度和宽度

**计算公式**：

<div align="left">$$H_{out} = \left\lfloor \frac{H + p_{top} + p_{bottom} - d_H \times (k_H - 1) - 1}{s_H} \right\rfloor + 1$$</div>

<div align="left">$$W_{out} = \left\lfloor \frac{W + p_{left} + p_{right} - d_W \times (k_W - 1) - 1}{s_W} \right\rfloor + 1$$</div>

### 1.6.2 数据载入流程

```

┌─────────────────────────────────────────────────────────────────────┐
│                           GM (DDR)                                  │
│    ┌──────────────┐    ┌──────────────┐    ┌──────────────┐         │
│    │ Feature Map  │    │    Weight    │    │    Bias      │         │
│    │   (agm)      │    │    (bgm)     │    │  (biasgm)    │         │
│    └──────────────┘    └──────────────┘    └──────────────┘         │
└─────────────────────────────────────────────────────────────────────┘
              │                    │                    │
              │ DataCopy           │ DataCopy           │ DataCopy
              │ (Dn2Nz/Nd2Nz)      │ (Dn2Nz/Nd2Nz)      │ (Dn2Nz)
              ▼                    ▼                    ▼
┌─────────────────────────────────────────────────────────────────────┐
│                           L1 Buffer                                 │
│    ┌──────────────┐    ┌──────────────┐    ┌──────────────┐         │
│    │     AL1      │    │     BL1      │    │   BiasL1     │         │
│    └──────────────┘    └──────────────┘    └──────────────┘         │
└─────────────────────────────────────────────────────────────────────┘
              │                    │                    │
              │ LoadData           │ LoadData           │ LoadData
              │ (Load3D)           │ (Load2D)           │ (Load2D)
              ▼                    ▼                    ▼
┌─────────────────────────────────────────────────────────────────────┐
│                           L0 Buffer                                 │
│    ┌──────────────┐    ┌──────────────┐    ┌──────────────┐         │
│    │     AL0      │    │     BL0      │    │   BiasBT     │         │
│    └──────────────┘    └──────────────┘    └──────────────┘         │
│              │                    │                    │            │
│              │     MMAD           │                    │            │
│              └────────────────────┼────────────────────┘            │
│                                   ▼                                 │
│    ┌────────────────────────────────────────────────────────┐       │
│    │                        L0C                             │       │
│    └────────────────────────────────────────────────────────┘       │
└─────────────────────────────────────────────────────────────────────┘
                                   │
                                   │ Fixpipe
                                   ▼
┌─────────────────────────────────────────────────────────────────────┐
│                      GM/UB (Output)                                 │
└─────────────────────────────────────────────────────────────────────┘

```

---

## 1.7 反向卷积

我们现在来进行卷积反向的数学推导
为了推导过程的严谨与清晰，我们设定以下数学符号：
我们设定以下三维张量符号：
- $X_{c, i, j}$：输入特征图。$c$ 为输入通道索引，$i, j$ 为空间坐标。
- $W_{c, m, n}$：卷积核（权重）。它也有 $C_{in}$ 个通道，每个通道的空间大小为 $m, n$。
- $Y_{i, j}$：正向输出特征图。因为只有一个滤波器，所以 $Y$ 是一个单通道的二维平面。
- $dY_{i, j}$：上一层回传的单通道误差梯度（即 $\frac{\partial L}{\partial Y_{i, j}}$）。

为了聚焦本质，我们假设步长（Stride）为 1，无填充（Padding=0），只存在一个卷积核(输出通道为1)。正向卷积的代数定义为：$$Y_{i, j} = \sum_{c} \sum_{m} \sum_{n} X_{c, i+m, j+n} \cdot W_{c, m, n}$$

<img src="./images/conv_2.svg" >

以该图卷积为例，$c=2,m=2,n=2$,$Y_{i, j}=56$即为每个通道下特征图左上角滑窗和卷积核乘积的累加

3.1 DW(权重梯度)
在反向传播中，算子ConvBackpropFilter 的任务是算出误差对每一个权重参数 $W_{c, m, n}$ 的偏导数
- 观察正向公式，结合上图我们可以发现一个权重$W_{c,m,n}$参与了所有输出结果$Y_{0,0},Y_{0,1},Y_{1,0},Y_{1,1}$的计算
因此，根据链式法则，我们需要分别计算每个输出点$Y_{i,j}$对$W_{c,m,n}$的梯度并累加
$$dW_{c, m, n} = \frac{\partial L}{\partial W_{c, m, n}} = \sum_{i} \sum_{j} \frac{\partial L}{\partial Y_{i, j}} \cdot \frac{\partial Y_{i, j}}{\partial W_{c, m, n}}$$

- 结合上图，我们来看${Y_{0,0}=56=W_{c=0,m=0,n=0} \cdot X_{c=0,m=0,n=0} + W_{c=0,m=0,n=1} \cdot X_{c=0,m=0,n=1} +……+ W_{c=1,m=1,n=1} \cdot X_{c=1,m=1,n=1}}$
我们计算$Y_{0,0}$对$W_{c=0,m=0,n=0}$的偏导，那么公式里除了$W_{c=0,m=0,n=0}$外的项都视为常量，所以$\frac{\partial Y_{0, 0}}{\partial W_{c=0, m=0, n=0}} = X_{c=0, m=0, n=0}$

以此类推$\frac{\partial Y_{0, 1}}{\partial W_{c=0, m=0, n=0}} = X_{c=0, m=0, n=1}$，$\frac{\partial Y_{1, 1}}{\partial W_{c=0, m=0, n=0}} = X_{c=0, m=1, n=1}$

最终可得：$$\frac{\partial Y_{i, j}}{\partial W_{c, m, n}} = X_{c, i+m, j+n}$$

- 结合上述两套公式可得：
$$dW_{c, m, n} = \sum_{i} \sum_{j} dY_{i, j} \cdot X_{c, i+m, j+n}$$

3.2 DX(输入梯度)
算子 ConvBackpropInput 的任务是算出误差对输入特征图 $X_{c, i, j}$ 的梯度。
- 一个特定的输入像素$X_{i,j}$在正向传播时参与卷积计算几次？假设卷积核大小为 $K \times K$，那么 $X_{i,j}$ 最多会参与 $K^2$ 个输出像素的计算。我们需要找出坐标之间的逆向映射关系。
如果正向是 $Y_{i_{out}, j_{out}} \leftarrow X_{i_{out}+m, j_{out}+n}$，令 $i = i_{out}+m$ 且 $j = j_{out}+n$，则受影响的输出坐标为 $i_{out} = i-m$，$j_{out} = j-n$。

我们继续依照上图举例，特征图第一个通道中间的数为5，分别在计算${Y_{0,0},Y_{0,1},Y_{1,0}，Y_{1,1}}$时和
${W_{1,1}, W_{1,0}, W_{0,1}, W_{0,0}}$做乘法，所以$dX_{c=0,i=1,j=1}=dY_{0,0} \cdot \frac{ \partial Y_{0, 0}}{\partial X_{c=0, i=1, j=1}} + dY_{1,0} \cdot \frac{ \partial Y_{1, 0}}{\partial X_{c=0, i=1, j=1}} + dY_{0,1} \cdot \frac{ \partial Y_{0, 1}}{\partial X_{c=0, i=1, j=1}} + dY_{1,1} \cdot \frac{ \partial Y_{1, 1}}{\partial X_{c=0, i=1, j=1}}$

可以推导得到：$$dX_{c, i, j} = \frac{\partial L}{\partial X_{c, i, j}} = \sum_{m} \sum_{n} \frac{\partial L}{\partial Y_{i-m, j-n}} \cdot \frac{\partial Y_{i-m, j-n}}{\partial X_{c, i, j}}$$

- 对特定输入像素求导，其系数是对应通道的权重：$$\frac{\partial Y_{i-m, j-n}}{\partial X_{c, i, j}} = W_{c, m, n}$$
- 得出最终公式 $$dX_{c, i, j} = \sum_{m} \sum_{n} dY_{i-m, j-n} \cdot W_{c, m, n}$$

---

## 1.8 扩展

### 1.8.1 Depthwise卷积

https://gitcode.com/cann/ops-nn/wiki/Conv2D算子Depthwise卷积开发与优化实践.md

### 1.8.2 3D卷积

3D卷积是2D卷积的自然扩展,在空间维度(H,W)基础上增加了D维度（Depth，深度）,2D卷积的尺寸$k_H \times k_W$，而3D的尺寸可以表示为$k_D \times k_H \times k_W$，3D卷积的具体计算方式与2D卷积类似，即滑动时与c个通道、尺寸大小为（depth，height，width）的图像做乘加运算，从而得到输出特征图中的一个值。可以表示为适用于视频处理、医学影像等三维数据场景。

#### 数学原理

**输入特征图**: $X \in \mathbb{R}^{N \times C_{in} \times D_{in} \times H_{in} \times W_{in}}$

**卷积核**: $K \in \mathbb{R}^{C_{out} \times C_{in} \times k_D \times k_H \times k_W}$

**输出特征图**: $Y \in \mathbb{R}^{N \times C_{out} \times D_{out} \times H_{out} \times W_{out}}$

**输出维度计算公式**:

<div align="left">$$D_{out} = \left\lfloor \frac{D_{in} + p_{head} + p_{tail} - d_D \times (k_D - 1) - 1}{s_D} \right\rfloor + 1$$</div>

<div align="left">$$H_{out} = \left\lfloor \frac{H_{in} + p_{top} + p_{bottom} - d_H \times (k_H - 1) - 1}{s_H} \right\rfloor + 1$$</div>

<div align="left">$$W_{out} = \left\lfloor \frac{W_{in} + p_{left} + p_{right} - d_W \times (k_W - 1) - 1}{s_W} \right\rfloor + 1$$</div>



---

## 1.9 章节测验

请根据本节课程学习内容完成以下题目进行自测。

**题目1**：关于3D卷积与2D卷积的区别，以下说法正确的是？

A. 3D卷积的卷积核形状为$k_H \times k_W$，2D卷积的卷积核形状为$k_D \times k_H \times k_W$

B. 3D卷积仅在空间维度滑动，2D卷积在空间和时间维度滑动

C. 3D卷积能够捕获时空联合特征，适用于视频动作识别任务

D. 3D卷积的计算量比2D卷积小

<br>


**题目2**：链式法则和偏导数在反向传播中的作用是？

A. 链式法则用于单个节点梯度计算，偏导数用于梯度跨层传递

B. 偏导数用于单个节点梯度计算，链式法则用于梯度跨层传递

C. 链式法则只用于前向传播，偏导数只用于反向传播

D. 链式法则和偏导数在反向传播中作用相同

<br>


**题目3**：给定输入特征图尺寸$1 \times 3 \times 32 \times 32$，卷积核$16 \times 3 \times 5 \times 5$，步长2，填充2，空洞率1，输出特征图尺寸为？

A. $1 \times 16 \times 15 \times 15$

B. $1 \times 16 \times 16 \times 16$

C. $1 \times 16 \times 17 \times 17$

D. $1 \times 64 \times 16 \times 16$

<br>

**题目4**：img2col的核心思想是？

A. 以时间换算力

B. 以空间换算力

C. 以精度换速度

D. 以内存换计算

<br>


---

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/09.02_answer.txt